# 1.2. Tải nguồn đa phiên bản, lọc tệp nhị phân và kiểm toán

## 1. Cơ chế tải tuần tự, nối tiếp HTTP `Range` và lọc trực tiếp trên luồng giải nén

- **Quản lý nhịp yêu cầu (`source_downloader.py`, `rate_limiter.py`)**: Mọi yêu cầu tải gói nguồn đi qua `https://export.arxiv.org/src/<id>vN` với `Accept-Encoding: identity` (tránh giải nén kép đối với gói `.tar.gz`), khóa đơn luồng đảm bảo khoảng nghỉ tối thiểu 3,0 giây tính từ thời điểm hoàn tất lượt tải trước và tuân thủ trường `Retry-After` từ máy chủ.
- **Tải nối tiếp qua HTTP `Range` (`206 Partial Content`) và sao lưu gia tăng**: Các gói lớn bị ngắt kết nối giữa chừng được lưu tạm vào tệp `.part` và đối chiếu `Content-Range` để tải tiếp phần còn thiếu; các gói nặng từ 20 MiB trở lên nếu gặp lỗi mạng lặp lại được dời về cuối phiên. Tiến độ thu thập được sao lưu định kỳ mỗi 25 bài thành 1.713 gói nén có kiểm tra mã băm SHA-256 (`drive_backup.py`).
- **Lọc bỏ hình ảnh ngay trong luồng đọc tar (`_safe_extract_tar`)**: Trình đọc duyệt trực tiếp từng thành phần của gói nén trong bộ nhớ đệm, cộng kích thước gốc vào `size_before_bytes`, kiểm tra chống thoát thư mục và chỉ ghi xuống đĩa các tệp văn bản thuộc tập cho phép `{.tex, .bib, .bbl}`, đồng thời giữ nguyên toàn bộ câu lệnh `\includegraphics` và môi trường `figure`/`table` bên trong tệp `.tex` để phục vụ Milestone 2.

## 2. Kết quả kiểm toán 41.498 phiên bản và dung lượng lưu trữ trước/sau khi lọc

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
sa = m1["source_audit"]
vs = m1["version_status"]
audit_rows = [
    ("Có mã nguồn TeX xác minh (tex_verified)", sa["tex_verified"], 100.0 * sa["tex_verified"] / scope["versions"], "Đủ tệp .tex/.bib/.bbl sạch cho bộ phân tích M2"),
    ("Chỉ nộp tệp PDF (pdf_only)", vs["pdf_only"], 100.0 * vs["pdf_only"] / scope["versions"], "Ngắt sớm sau khi kiểm tra chữ ký %PDF- ở đầu luồng"),
    ("Gói nguồn không chứa tệp .tex (no_latex)", vs["no_latex"], 100.0 * vs["no_latex"] / scope["versions"], "Thuộc nhóm empty_verified (2.756 + 8 = 2.764 phiên bản)"),
    ("Máy chủ arXiv trả về HTTP 404 (not_found_verified)", sa["not_found_verified"], 100.0 * sa["not_found_verified"] / scope["versions"], "Nguồn bị tác giả rút hoặc không công khai trên arXiv"),
    ("Còn chờ hoặc lỗi tải nguồn (source_pending_or_failed)", sa["source_pending_or_failed"], 100.0 * sa["source_pending_or_failed"] / scope["versions"], "11 phiên bản cuối phiên thuộc phân vùng TRIET"),
]
df_aud = pd.DataFrame(audit_rows, columns=["Phân loại kiểm toán phiên bản", "Số phiên bản", "Tỷ lệ (%)", "Ghi chú kiểm toán"])
display(style_table(
    df_aud,
    caption="Bảng 2.1. Kết quả kiểm toán độc lập trên 41.498 phiên bản (xác minh hợp lệ 41.487/41.498 phiên bản, đạt 99,97%).",
    int_cols=["Số phiên bản"],
    pct_cols=["Tỷ lệ (%)"],
))

tot_before_b = m1["total_source_size_before_bytes"]
tot_after_b = m1["total_source_size_after_bytes"]
saved_b = tot_before_b - tot_after_b
df_size = pd.DataFrame([
    ("Kích thước trung bình 1 bài trước lọc ảnh (22.748 bài có TeX)", f"{fmt_num(m1['mean_before_bytes'], 2)} byte", f"{fmt_num(m1['mean_before_bytes']/1048576, 3)} MiB/bài"),
    ("Kích thước trung bình 1 bài sau lọc chỉ giữ .tex, .bib, .bbl", f"{fmt_num(m1['mean_after_bytes'], 2)} byte", f"{fmt_num(m1['mean_after_bytes']/1048576, 3)} MiB/bài"),
    ("Kích thước trung bình tính trên toàn tập 25.000 bài (trước và sau)", f"{fmt_num(tot_before_b/scope['papers'], 2)} còn {fmt_num(tot_after_b/scope['papers'], 2)} byte", f"{fmt_num(tot_before_b/scope['papers']/1048576, 3)} còn {fmt_num(tot_after_b/scope['papers']/1048576, 3)} MiB/bài"),
    ("Tổng dung lượng toàn kho 25.000 bài trước khi lọc", f"{fmt_int(tot_before_b)} byte", f"{fmt_num(tot_before_b/1024**3, 2)} GiB ({fmt_num(tot_before_b/1e9, 2)} GB)"),
    ("Tổng dung lượng toàn kho 25.000 bài sau khi lọc", f"{fmt_int(tot_after_b)} byte", f"{fmt_num(tot_after_b/1024**3, 2)} GiB ({fmt_num(tot_after_b/1e9, 2)} GB)"),
    ("Dung lượng tệp nhị phân loại bỏ được (tỷ lệ giảm)", f"{fmt_int(saved_b)} byte", f"{fmt_num(saved_b/1024**3, 2)} GiB (-{fmt_pct(m1['source_reduction'], 2, True)})"),
], columns=["Chỉ tiêu dung lượng mã nguồn", "Giá trị đo (Bytes)", "Quy đổi (MiB / GiB)"])
display(style_table(df_size, caption="Bảng 2.2. Kích thước trung bình bài báo tính bằng byte và tổng dung lượng trước/sau khi lọc bỏ hình ảnh."))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8), gridspec_kw={"width_ratios": [1.25, 1.0]})
y_a = np.arange(len(df_aud))
ax1.barh(y_a, df_aud["Số phiên bản"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], height=0.54)
ax1.set_yticks(y_a)
ax1.set_yticklabels(["TeX xác minh", "Chỉ có PDF", "Không có .tex", "HTTP 404", "Chờ tải nguồn"])
ax1.invert_yaxis()
ax1.set_xscale("symlog", linthresh=30)
ax1.set_xlim(0, 120000)
ax1.grid(axis="x")
ax1.set_xlabel("Số phiên bản (thang symlog)")
ax1.set_title("Hình 2.1a. Kiểm toán 41.498 phiên bản nguồn")
for i_a, r_a in enumerate(df_aud.to_dict("records")):
    ax1.text(float(r_a["Số phiên bản"]) * 1.16 + 2, i_a, f"{fmt_int(r_a['Số phiên bản'])} ({fmt_pct(r_a['Tỷ lệ (%)'])})",
             va="center", fontsize=8.0)

sx = np.arange(2)
bars_s = ax2.bar(sx, [m1["mean_before_bytes"] / 1e6, m1["mean_after_bytes"] / 1e6],
                 color=[BOOK_PALETTE["slate"], BOOK_PALETTE["navy"]], width=0.48)
ax2.set_xticks(sx)
ax2.set_xticklabels(["Trước lọc ảnh\n(gói nguồn gốc)", "Sau lọc giữ\n.tex / .bib / .bbl"])
ax2.set_ylabel("Triệu byte / bài báo (MB)")
ax2.set_ylim(0, 12.0)
ax2.grid(axis="y")
ax2.set_title("Hình 2.1b. Giảm 89,30% kích thước trung bình bài báo")
ax2.text(bars_s[0].get_x() + bars_s[0].get_width() / 2, 10.2,
         f"{fmt_int(m1['mean_before_bytes'])} B\n(9,484 MiB)", ha="center", fontsize=8.1)
ax2.text(bars_s[1].get_x() + bars_s[1].get_width() / 2, 1.4,
         f"{fmt_int(m1['mean_after_bytes'])} B\n(1,015 MiB, -89,30%)",
         ha="center", fontsize=8.1, fontweight="bold", color=BOOK_PALETTE["navy"])
plt.tight_layout()
plt.show()

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/source_downloader.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/source_downloader.py).
- Mã nguồn triển khai: [`src/crawler/retry_sources.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/retry_sources.py).
- Mã nguồn triển khai: [`src/crawler/validate_sources.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/validate_sources.py).
- Mã nguồn triển khai: [`src/crawler/drive_backup.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/drive_backup.py).